# OpenAI Embedding Evaluation – Verlan ↔ Base Form Benchmark

## Evaluation design

This notebook benchmarks OpenAI text embeddings on the task of mapping French **verlan forms** to their **standard (base) forms**.

### Steps

1. **Load data** – read `verlan_base_forms.csv`, keep valid pairs, deduplicate.
2. **Embed** – call OpenAI Embeddings API for every unique word; cache results locally so the API is never called twice.
3. **Pairwise similarity** – compute cosine similarity for every true (verlan, base) pair and print summary statistics.
4. **Retrieval benchmark** – for each verlan form, rank all candidate base forms and compute Acc@1, Acc@5, MRR, and Average Rank.
5. **Random-negative baseline** – compare true-pair cosine similarity against randomly mismatched pairs.
6. **Nearest-neighbour analysis** – for a set of probe words, show the 10 nearest neighbours in embedding space.

All outputs are saved to disk (CSV / JSON) for later comparison with the fastText baseline.

In [ ]:
import os

# Set your OpenAI API key here (or export it in your shell before launching Jupyter)
os.environ["OPENAI_API_KEY"] = ""  # ← replace with your actual key

In [ ]:
# ── Imports & configuration ───────────────────────────────────────────────────
import os
import json
import pickle
import random
import numpy as np
import pandas as pd
from pathlib import Path
from openai import OpenAI

# ── Reproducibility ───────────────────────────────────────────────────────────
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# ── Paths (all relative to this notebook) ────────────────────────────────────
# BASE_DIR       = Path("llm_verlan")
DATA_CSV       = "../data/processed/verlan_database.csv"
CACHE_FILE     = Path("../artifacts/cache/openai_embeddings_cache.pkl")  # local embedding cache
PAIRWISE_CSV   = Path("../outputs/openai_pairwise_results.csv")
RETRIEVAL_CSV  = Path("../outputs/openai_retrieval_results.csv")
NEIGHBORS_JSON = Path("../outputs/openai_neighbors.json")

# ── Model configuration ───────────────────────────────────────────────────────
EMBEDDING_MODEL = "text-embedding-3-large"  # swap to text-embedding-3-large if desired
BATCH_SIZE      = 100                        # words per API request

# ── Probe words for nearest-neighbour analysis ────────────────────────────────
PROBE_WORDS = ["meuf", "teuf", "keuf", "ouf"]

# ── OpenAI client (reads OPENAI_API_KEY from environment automatically) ───────
client = OpenAI()  # raises AuthenticationError if OPENAI_API_KEY is not set
print("Configuration loaded. Model:", EMBEDDING_MODEL)

Configuration loaded. Model: text-embedding-3-small


In [14]:
# ── 1. Load data ──────────────────────────────────────────────────────────────

def load_pairs(csv_path: Path) -> pd.DataFrame:
    """Load verlan↔base pairs from CSV and clean them."""
    df = pd.read_csv(csv_path, encoding="utf-8")

    # Keep only the two columns we need
    df = df[["canonical_verlan", "base_forms"]].copy()

    # Drop rows where either column is missing or empty
    df = df.dropna(subset=["canonical_verlan", "base_forms"])
    df["canonical_verlan"] = df["canonical_verlan"].str.strip()
    df["base_forms"]   = df["base_forms"].str.strip()
    df = df[df["canonical_verlan"].str.len() > 0]
    df = df[df["base_forms"].str.len() > 0]

    # Remove duplicates
    df = df.drop_duplicates(subset=["canonical_verlan", "base_forms"]).reset_index(drop=True)
    return df


pairs_df = load_pairs(DATA_CSV)
print(f"Usable verlan-base pairs: {len(pairs_df)}")
pairs_df.head(100)

Usable verlan-base pairs: 53


,canonical_verlan,base_forms
0,meuf,femme
1,çacomme,comme ça
2,ken,niquer
3,teuf,fête
4,barjo,jobard
5,zarbi,bizarre
6,vénère,énervé
7,resta,star de star
8,tej,jeter
9,reum,mère


In [15]:
# ── 2. OpenAI embeddings with local caching ───────────────────────────────────

def _embed_batch(words: list[str]) -> list[list[float]]:
    """Call OpenAI Embeddings API for a batch of words."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=words,
        encoding_format="float",
    )
    # Results are returned in the same order as input
    return [item.embedding for item in sorted(response.data, key=lambda x: x.index)]


def get_embeddings(words: list[str], cache_file: Path) -> dict[str, np.ndarray]:
    """
    Return a {word: embedding_vector} dict.
    Loads whatever is cached, only calls the API for uncached words,
    then saves the updated cache to disk.
    """
    # Load existing cache
    cache: dict[str, np.ndarray] = {}
    if cache_file.exists():
        with open(cache_file, "rb") as f:
            cache = pickle.load(f)
        print(f"Cache loaded: {len(cache)} words already embedded.")

    # Find which words still need embedding
    missing = [w for w in words if w not in cache]
    if missing:
        print(f"Fetching embeddings for {len(missing)} new words "
              f"in batches of {BATCH_SIZE}…")
        for i in range(0, len(missing), BATCH_SIZE):
            batch = missing[i : i + BATCH_SIZE]
            vectors = _embed_batch(batch)
            for word, vec in zip(batch, vectors):
                cache[word] = np.array(vec, dtype=np.float32)
            print(f"  {min(i + BATCH_SIZE, len(missing))}/{len(missing)} done")

        # Persist updated cache
        with open(cache_file, "wb") as f:
            pickle.dump(cache, f)
        print(f"Cache saved to {cache_file}.")
    else:
        print("All words already cached – no API calls needed.")

    return cache


# Collect all unique words from both columns
all_words = list(
    set(pairs_df["canonical_verlan"].tolist()) | set(pairs_df["base_forms"].tolist())
)
print(f"Unique words to embed: {len(all_words)}")

embeddings = get_embeddings(all_words, CACHE_FILE)
print(f"Embeddings ready for {len(embeddings)} words.")

Unique words to embed: 100
Cache loaded: 477 words already embedded.
Fetching embeddings for 4 new words in batches of 100…
  4/4 done
Cache saved to ../artifacts/cache/openai_embeddings_cache.pkl.
Embeddings ready for 481 words.


In [17]:
# ── 3. Pairwise cosine similarity ─────────────────────────────────────────────

def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors."""
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


def compute_pairwise(df: pd.DataFrame, emb: dict[str, np.ndarray]) -> pd.DataFrame:
    """Compute cosine similarity for every true (verlan, base) pair."""
    rows = []
    for _, row in df.iterrows():
        v, b = row["canonical_verlan"], row["base_forms"]
        if v in emb and b in emb:
            sim = cosine_similarity(emb[v], emb[b])
            rows.append({"verlan_form": v, "base_form": b, "cosine_similarity": sim})
    return pd.DataFrame(rows)


pairwise_df = compute_pairwise(pairs_df, embeddings)
pairwise_df.to_csv(PAIRWISE_CSV, index=False, encoding="utf-8")
print(f"Pairwise results saved to {PAIRWISE_CSV}  ({len(pairwise_df)} pairs)\n")

sims = pairwise_df["cosine_similarity"]
print("── Cosine similarity summary ─────────────────────────")
print(f"  Mean   : {sims.mean():.4f}")
print(f"  Median : {sims.median():.4f}")
print(f"  Min    : {sims.min():.4f}")
print(f"  Max    : {sims.max():.4f}")

print("\n── Top 10 highest-scoring pairs ──────────────────────")
print(pairwise_df.nlargest(10, "cosine_similarity").to_string(index=False))

print("\n── Top 10 lowest-scoring pairs ───────────────────────")
print(pairwise_df.nsmallest(10, "cosine_similarity").to_string(index=False))

Pairwise results saved to ../outputs/openai_pairwise_results.csv  (53 pairs)

── Cosine similarity summary ─────────────────────────
  Mean   : 0.3789
  Median : 0.3658
  Min    : 0.2073
  Max    : 0.6350

── Top 10 highest-scoring pairs ──────────────────────
verlan_form base_form  cosine_similarity
        zen       nez           0.635004
    çacomme  comme ça           0.621271
       feuj      juif           0.586535
       meuf     femme           0.581870
    loubard   balourd           0.567714
      deouf       fou           0.524853
     chelou    louche           0.501205
      pécho    choper           0.481209
    chichon  haschich           0.471653
       zyva     vas-y           0.459791

── Top 10 lowest-scoring pairs ───────────────────────
verlan_form    base_form  cosine_similarity
       zinc       cousin           0.207269
      zarbi      bizarre           0.228709
       reup         père           0.229543
       keum          mec           0.229911
       reum 

In [20]:
# ── 4. Retrieval benchmark ────────────────────────────────────────────────────
# For each verlan form, rank ALL candidate base forms by cosine similarity
# and measure how high the true base form appears in the ranking.

def retrieval_benchmark(
    df: pd.DataFrame,
    emb: dict[str, np.ndarray],
) -> tuple[pd.DataFrame, dict]:
    """
    For each verlan form, rank all unique base forms by cosine similarity.
    Returns per-example results and aggregate metrics.
    """
    unique_bases = list(df["base_forms"].unique())

    # Pre-compute normalised base vectors for fast dot-product ranking
    base_matrix = np.stack([emb[b] / (np.linalg.norm(emb[b]) + 1e-10) for b in unique_bases])

    # True base form per verlan (one-to-one after dedup; keep first if multiple)
    true_base = df.drop_duplicates("canonical_verlan").set_index("canonical_verlan")["base_forms"]

    rows = []
    reciprocal_ranks = []
    ranks = []
    top1_hits = []
    top5_hits = []

    for verlan, base in true_base.items():
        if verlan not in emb or base not in emb:
            continue

        # Cosine scores against all candidate base forms
        v_vec = emb[verlan]
        v_norm = v_vec / (np.linalg.norm(v_vec) + 1e-10)
        scores = base_matrix @ v_norm  # shape (n_bases,)

        # Rank (1-indexed, lowest rank = best)
        order = np.argsort(-scores)  # descending
        ranked_bases = [unique_bases[i] for i in order]

        rank = ranked_bases.index(base) + 1 if base in ranked_bases else len(ranked_bases) + 1
        top5 = ranked_bases[:5]

        rows.append({
            "verlan_form":       verlan,
            "true_base_form":    base,
            "rank":              rank,
            "top1_prediction":   top5[0] if len(top5) >= 1 else "",
            "top2_prediction":   top5[1] if len(top5) >= 2 else "",
            "top3_prediction":   top5[2] if len(top5) >= 3 else "",
            "top4_prediction":   top5[3] if len(top5) >= 4 else "",
            "top5_prediction":   top5[4] if len(top5) >= 5 else "",
        })
        reciprocal_ranks.append(1.0 / rank)
        ranks.append(rank)
        top1_hits.append(int(rank == 1))
        top5_hits.append(int(rank <= 5))

    results_df = pd.DataFrame(rows)

    metrics = {
        "acc@1":      np.mean(top1_hits),
        "acc@5":      np.mean(top5_hits),
        "mrr":        np.mean(reciprocal_ranks),
        "avg_rank":   np.mean(ranks),
        "n_examples": len(rows),
        "n_bases":    len(unique_bases),
    }
    return results_df, metrics


retrieval_df, metrics = retrieval_benchmark(pairs_df, embeddings)
retrieval_df.to_csv(RETRIEVAL_CSV, index=False, encoding="utf-8")
print(f"Retrieval results saved to {RETRIEVAL_CSV}\n")

print("── Retrieval metrics ─────────────────────────────────")
print(f"  Candidates (base forms) : {metrics['n_bases']}")
print(f"  Examples evaluated      : {metrics['n_examples']}")
print(f"  Acc@1                   : {metrics['acc@1']:.4f}")
print(f"  Acc@5                   : {metrics['acc@5']:.4f}")
print(f"  MRR                     : {metrics['mrr']:.4f}")
print(f"  Average Rank            : {metrics['avg_rank']:.2f}")

Retrieval results saved to ../outputs/openai_retrieval_results.csv

── Retrieval metrics ─────────────────────────────────
  Candidates (base forms) : 47
  Examples evaluated      : 53
  Acc@1                   : 0.2830
  Acc@5                   : 0.6226
  MRR                     : 0.4426
  Average Rank            : 6.49


In [23]:
# ── 5. Random negative baseline ───────────────────────────────────────────────
# For each verlan form, sample a *wrong* base form and compare similarities.
# A well-trained embedding model should score true pairs higher than random ones.

def random_negative_baseline(
    df: pd.DataFrame,
    emb: dict[str, np.ndarray],
    seed: int = RANDOM_SEED,
) -> None:
    rng = random.Random(seed)
    all_bases = df["base_forms"].unique().tolist()

    true_sims  = []
    neg_sims   = []

    for _, row in df.iterrows():
        v, b = row["canonical_verlan"], row["base_forms"]
        if v not in emb or b not in emb:
            continue

        # True pair similarity
        true_sims.append(cosine_similarity(emb[v], emb[b]))

        # Sample a wrong base form (ensure it is different from the true one)
        candidates = [x for x in all_bases if x != b]
        if not candidates:
            continue
        neg_b = rng.choice(candidates)
        if neg_b in emb:
            neg_sims.append(cosine_similarity(emb[v], emb[neg_b]))

    print("── Random negative baseline ──────────────────────────")
    print(f"  Avg cosine (true pairs)   : {np.mean(true_sims):.4f}  (n={len(true_sims)})")
    print(f"  Avg cosine (random pairs) : {np.mean(neg_sims):.4f}  (n={len(neg_sims)})")
    delta = np.mean(true_sims) - np.mean(neg_sims)
    print(f"  Δ (true − random)         : {delta:+.4f}")
    if delta > 0:
        print("  → True pairs score HIGHER than random – embeddings carry signal.")
    else:
        print("  → True pairs do NOT score higher – embeddings may not help here.")


random_negative_baseline(pairs_df, embeddings)

── Random negative baseline ──────────────────────────
  Avg cosine (true pairs)   : 0.3789  (n=53)
  Avg cosine (random pairs) : 0.2575  (n=53)
  Δ (true − random)         : +0.1214
  → True pairs score HIGHER than random – embeddings carry signal.


In [24]:
# ── 6. Nearest-neighbour analysis ────────────────────────────────────────────
# For each probe word, find the 10 closest words in the benchmark vocabulary.

def nearest_neighbors(
    probe_words: list[str],
    emb: dict[str, np.ndarray],
    top_k: int = 10,
) -> dict[str, list[dict]]:
    """
    Return {probe_word: [{word, cosine_similarity}, ...]} for each probe.
    The probe word itself is excluded from its own neighbour list.
    """
    vocab = list(emb.keys())
    vocab_matrix = np.stack([emb[w] / (np.linalg.norm(emb[w]) + 1e-10) for w in vocab])

    results: dict[str, list[dict]] = {}

    for probe in probe_words:
        if probe not in emb:
            print(f"  '{probe}' not found in vocabulary – skipping.")
            results[probe] = []
            continue

        p_vec = emb[probe]
        p_norm = p_vec / (np.linalg.norm(p_vec) + 1e-10)
        scores = vocab_matrix @ p_norm

        # Exclude the probe word itself
        order = np.argsort(-scores)
        neighbors = []
        for idx in order:
            if vocab[idx] == probe:
                continue
            neighbors.append({"word": vocab[idx], "cosine_similarity": float(scores[idx])})
            if len(neighbors) == top_k:
                break

        results[probe] = neighbors

    return results


nn_results = nearest_neighbors(PROBE_WORDS, embeddings, top_k=10)

# Print results
for probe, neighbors in nn_results.items():
    print(f"\n── Nearest neighbours of '{probe}' ───────────────────")
    if not neighbors:
        print("  (no results)")
    for rank, nb in enumerate(neighbors, 1):
        print(f"  {rank:2d}.  {nb['word']:<20s}  {nb['cosine_similarity']:.4f}")

# Save to JSON (convert numpy floats → Python floats for JSON serialisation)
with open(NEIGHBORS_JSON, "w", encoding="utf-8") as f:
    json.dump(nn_results, f, ensure_ascii=False, indent=2)
print(f"\nNearest-neighbour results saved to {NEIGHBORS_JSON}.")

  'ouf' not found in vocabulary – skipping.

── Nearest neighbours of 'meuf' ───────────────────
   1.  meufeu                0.8495
   2.  méfu                  0.6416
   3.  merdeuf               0.5961
   4.  beurette              0.5927
   5.  femme                 0.5819
   6.  keuf                  0.5642
   7.  teufé                 0.5640
   8.  feuj                  0.5597
   9.  seuf                  0.5438
  10.  peufra                0.5376

── Nearest neighbours of 'teuf' ───────────────────
   1.  teufé                 0.9030
   2.  treuf                 0.6528
   3.  keuf                  0.6160
   4.  seuf                  0.5912
   5.  teufer                0.5648
   6.  teuch                 0.5565
   7.  téci                  0.5469
   8.  meuf                  0.5336
   9.  tebé                  0.5303
  10.  téco                  0.5217

── Nearest neighbours of 'keuf' ───────────────────
   1.  seuf                  0.6721
   2.  teuf                  0.6160
   3.

## How to run

1. **Install dependencies**
   ```bash
   pip install openai numpy pandas
   ```

2. **Set your API key**
   ```bash
   export OPENAI_API_KEY="sk-..."
   ```
   Or create a `.env` file and load it with `python-dotenv`.

3. **Run the notebook top-to-bottom.** On the first run the API is called once to embed all unique words; subsequent runs load everything from `openai_embeddings_cache.pkl` — zero extra API cost.

4. **Output files produced**

   | File | Contents |
   |------|----------|
   | `openai_embeddings_cache.pkl` | Cached word vectors |
   | `openai_pairwise_results.csv` | Per-pair cosine similarities |
   | `openai_retrieval_results.csv` | Per-verlan rank + top-5 predictions |
   | `openai_neighbors.json` | Nearest neighbours for probe words |

---

## API cost minimisation notes

- **`text-embedding-3-small`** costs ~$0.02 per million tokens. A typical French word is ≈ 1–2 tokens, so embedding 1 000 unique words ≈ **< $0.0001**.
- The **local pickle cache** (`openai_embeddings_cache.pkl`) ensures you pay only once per unique word, even across notebook restarts.
- **Batching** (100 words / request) reduces HTTP overhead; the limit is 2 048 inputs per call.
- If you later extend the word list, only the newly added words trigger API calls — the rest are served from cache.
- To minimise cost further, avoid `text-embedding-3-large` (5× more expensive) unless `text-embedding-3-small` retrieval metrics are insufficient.